# 21 — Paper Figures: Augmentation Trends and Baseline Comparison

Two figures replacing/adding to notebook 18's outputs for the final LaTeX pass.

**(a) Augmentation trends across configurations** (`augmentation_trends.pdf`) — replaces
`robustness.pdf` (former Figure 8). Instead of one small-multiples panel per classifier at a
single fixed configuration (500 NSEP), this shows all four augmentation methods (SMOTE, ADASYN,
TimeGAN, Diffusion), pooled across all four classifiers, across their four shared configurations
(Balanced, 8000, 2000, 500 retained NSEP). One panel instead of four, and it directly visualizes
the trend already discussed in prose (Section 4.4, "Impacts of Over Sampling and RUS"): TSS rises
monotonically for every method as the retained NSEP count shrinks, and classical methods
(solid lines) stay above generative methods (dashed lines) at every configuration.

**(b) Best result vs. published baselines** (`baseline_comparison.pdf`) — new figure for the
"Comparison with Baselines" section. Horizontal grouped bars (TSS / POD / Accuracy) for our best
GRU configuration against the five literature methods in Table 9, annotated with each entry's
class imbalance ratio so the reader can see the operational realism gap alongside the skill
comparison. Literature TSS/POD/Accuracy values are the same ones already used in Table 9 (quoted
from the original papers). Imbalance ratios were independently researched per source paper (see
citations in the code comments below); the Bi-LSTM ratio is flagged with `*` because it is derived
from the paper's reported SEP-event-day count rather than a ratio the paper states directly.

**Reads:** `./results/*.txt` (panel a only; panel b's literature numbers are hard-coded from the
cited papers, matching Table 9 in `paper/main.tex`).
**Writes:** `./paper/Figures/augmentation_trends.pdf/.png` and
`./paper/Figures/baseline_comparison.pdf/.png`.

## 1. Shared Style

In [1]:
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

RESULTS_DIR = "./results"
FIG_DIR = "./paper/Figures"
os.makedirs(FIG_DIR, exist_ok=True)

plt.rcParams.update({
    "font.family": "serif", "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "dejavuserif",
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.4, "ytick.major.size": 2.4,
    "xtick.labelsize": 6.4, "ytick.labelsize": 6.6,
    "axes.labelsize": 7.0, "axes.titlesize": 7.4, "legend.fontsize": 6.2,
    "axes.edgecolor": "0.35", "text.color": "0.0",
    "axes.labelcolor": "0.0", "xtick.color": "0.25", "ytick.color": "0.25",
})

BLUE, VERM, TEAL, GOLD = "#0072B2", "#D55E00", "#009E73", "#CC9A00"
CLASSIFIERS = ["gru", "patchtst", "svm", "inceptiontime"]
COLUMN = {"TP": 0, "TN": 1, "FP": 2, "FN": 3, "tss": 4}

def load_col(clf, key, col=COLUMN["tss"]):
    fp = os.path.join(RESULTS_DIR, f"{clf}_{key}.txt")
    return np.array([float(l.split(",")[col]) for l in open(fp) if l.strip()])

def load_all(key, col=COLUMN["tss"]):
    out = []
    for clf in CLASSIFIERS:
        out.extend(load_col(clf, key, col).tolist())
    return np.array(out)

print("Style + loader ready.")

Style + loader ready.


## 2. Figure (a) — Augmentation Trends Across Configurations

In [2]:
CONFIGS = ["Balanced", "8000", "2000", "500"]
METHODS = {
    "ADASYN":    {"Balanced": "tomek_adasyn_balanced",  "8000": "tomek_rus8000_adasyn8000",
                  "2000": "tomek_rus2000_adasyn2000",    "500": "tomek_rus500_adasyn500"},
    "SMOTE":     {"Balanced": "tomek_smote_balanced",    "8000": "tomek_rus8000_smote8000",
                  "2000": "tomek_rus2000_smote2000",     "500": "tomek_rus500_smote500"},
    "TimeGAN":   {"Balanced": "timegan_balanced", "8000": "timegan_8000",
                  "2000": "timegan_2000",          "500": "timegan_500"},
    "Diffusion": {"Balanced": "diffusion_balanced", "8000": "diffusion_8000",
                  "2000": "diffusion_2000",          "500": "diffusion_500"},
}
METHOD_STYLE = {
    "ADASYN":    dict(color=TEAL, ls="-",  marker="o"),
    "SMOTE":     dict(color=BLUE, ls="-",  marker="s"),
    "TimeGAN":   dict(color=VERM, ls="--", marker="^"),
    "Diffusion": dict(color=GOLD, ls="--", marker="D"),
}
X_NSEP = {"Balanced": 12323, "8000": 8000, "2000": 2000, "500": 500}
xs = np.arange(len(CONFIGS))

means = {m: [] for m in METHODS}
stds = {m: [] for m in METHODS}
for m, keys in METHODS.items():
    for cfg in CONFIGS:
        v = load_all(keys[cfg])
        means[m].append(v.mean())
        stds[m].append(v.std())

print(f"{'method':<10}", *[f"{c:>10}" for c in CONFIGS])
for m in METHODS:
    print(f"{m:<10}", *[f"{v:>10.3f}" for v in means[m]])

method       Balanced       8000       2000        500
ADASYN          0.259      0.305      0.440      0.452
SMOTE           0.270      0.262      0.382      0.449
TimeGAN         0.020      0.026      0.081      0.275
Diffusion       0.102      0.099      0.182      0.323


In [3]:
# FIG -- augmentation trends: pooled mean TSS by method across shared configurations
fig, ax = plt.subplots(figsize=(3.4, 2.85))

for m in ["ADASYN", "SMOTE", "TimeGAN", "Diffusion"]:
    mu = np.array(means[m]); sd = np.array(stds[m])
    st = METHOD_STYLE[m]
    ax.plot(xs, mu, color=st["color"], ls=st["ls"], marker=st["marker"],
            ms=3.6, lw=1.1, label=m, zorder=4)
    ax.fill_between(xs, mu - sd, mu + sd, color=st["color"], alpha=0.12, lw=0, zorder=2)

ax.set_xticks(xs)
ax.set_xticklabels([f"{c}\n({X_NSEP[c]:,} NSEP)" for c in CONFIGS], fontsize=5.8)
ax.set_ylabel("Pooled mean TSS (4 classifiers)")
ax.set_xlabel("Configuration (decreasing retained NSEP)")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
ax.grid(True, axis="y", color="0.88", lw=0.5, zorder=0)
ax.set_axisbelow(True)
ax.set_ylim(-0.02, 0.78)

ax.legend(frameon=False, loc="upper left", fontsize=6.0, handlelength=1.6,
          handletextpad=0.5, borderpad=0.1, labelspacing=0.3, ncol=2,
          columnspacing=1.0, bbox_to_anchor=(-0.02, 1.10))

ax.text(0.98, 0.97,
        "classical (solid) $>$ generative (dashed)\nat every configuration",
        transform=ax.transAxes, ha="right", va="top", fontsize=5.6,
        color="0.35", style="italic")

fig.tight_layout()
fig.savefig(f"{FIG_DIR}/augmentation_trends.pdf")
fig.savefig(f"{FIG_DIR}/augmentation_trends.png", dpi=340)
plt.show()
print("Saved augmentation_trends.pdf/png")

Saved augmentation_trends.pdf/png


/var/folders/fx/gjhbmrbj5jn295_9wrqpbsv80000gn/T/ipykernel_47066/1966890525.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Figure (b) — Best Result vs. Published Baselines

In [4]:
# Literature TSS/POD/Accuracy match Table 9 in paper/main.tex (quoted from the original papers).
# Imbalance ratios match Fig. "related_work" in SEP_DataAugmentation 2's own paper
# (../SEP_DataAugmentation 2/paper/figures/related_work.pdf, its fig:related), which
# independently sourced the NonSEP:SEP test-partition ratio for each of these same five
# literature baselines (labeled directly inside that figure's bars):
#  - SEPNET-TS : 3:1     - SMARP SVM : 1:1      - Bi-LSTM : 42:1
#  - UDM       : 1:1     - MEMPSEP-I : 2:1
rows = [
    ("SEPNET-TS",         0.450, 0.560, 0.800, "3:1",   False),
    ("SMARP SVM",         0.470, 0.690, 0.720, "1:1",   False),
    ("Bi-LSTM",           0.530, 0.620, 0.990, "42:1",  False),
    ("UDM",               0.590, 0.800, 0.800, "1:1",   False),
    ("MEMPSEP-I",         0.630, 0.830, 0.810, "2:1",   False),
    ("This work\n(GRU)",  0.671, 0.794, 0.876, "104:1", True),
]

labels = [r[0] for r in rows]
tss = np.array([r[1] for r in rows])
pod = np.array([r[2] for r in rows])
acc = np.array([r[3] for r in rows])
ratio = [r[4] for r in rows]
ours = [r[5] for r in rows]
print(f"{'model':<16} {'TSS':>6} {'POD':>6} {'Accuracy':>9}  {'ratio':>10}")
for r in rows:
    print(f"{r[0].replace(chr(10),' '):<16} {r[1]:>6.3f} {r[2]:>6.3f} {r[3]:>9.3f}  {r[4]:>10}")

model               TSS    POD  Accuracy       ratio
SEPNET-TS         0.450  0.560     0.800         3:1
SMARP SVM         0.470  0.690     0.720         1:1
Bi-LSTM           0.530  0.620     0.990        42:1
UDM               0.590  0.800     0.800         1:1
MEMPSEP-I         0.630  0.830     0.810         2:1
This work (GRU)   0.671  0.794     0.876       104:1


In [5]:
# FIG -- best result vs. published baselines, with imbalance ratio annotated per row
y = np.arange(len(rows))
h = 0.24

fig, ax = plt.subplots(figsize=(3.55, 3.35))

ax.barh(y + h, tss, height=h, color=TEAL, edgecolor="none", zorder=3, label="TSS")
ax.barh(y,      pod, height=h, color=BLUE, edgecolor="none", zorder=3, label="POD")
ax.barh(y - h,  acc, height=h, color="0.68", edgecolor="none", zorder=3, label="Accuracy")

for vals, yoff, col in [(tss, h, TEAL), (pod, 0, BLUE), (acc, -h, "0.35")]:
    for yi, v in zip(y, vals):
        ax.text(v + 0.015, yi + yoff, f"{v:.2f}", ha="left", va="center",
                fontsize=5.4, color=col, zorder=4)

for yi, is_ours in zip(y, ours):
    if is_ours:
        ax.axhspan(yi - 1.5 * h, yi + 1.5 * h, color=GOLD, alpha=0.10, zorder=1)

for yi, r in zip(y, ratio):
    ax.text(1.16, yi, r, transform=ax.get_yaxis_transform(),
            ha="left", va="center", fontsize=6.0, color="0.25")
ax.text(1.16, len(rows) - 0.15, "Imbalance\nratio", transform=ax.get_yaxis_transform(),
        ha="left", va="bottom", fontsize=5.8, color="0.35", style="italic", linespacing=1.1)

ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=6.8)
for tick, is_ours in zip(ax.get_yticklabels(), ours):
    if is_ours:
        tick.set_color(GOLD)
        tick.set_fontweight("bold")

ax.set_xlim(0, 1.05)
ax.set_xlabel("Score")
ax.invert_yaxis()
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
ax.grid(True, axis="x", color="0.90", lw=0.5, zorder=0)
ax.set_axisbelow(True)

ax.legend(loc="lower center", frameon=False, fontsize=6.4, handlelength=1.1,
          handletextpad=0.4, borderpad=0.1, labelspacing=0.25, ncol=3,
          columnspacing=1.2, bbox_to_anchor=(0.42, 1.01))

fig.tight_layout()
fig.savefig(f"{FIG_DIR}/baseline_comparison.pdf", bbox_inches="tight")
fig.savefig(f"{FIG_DIR}/baseline_comparison.png", dpi=340, bbox_inches="tight")
plt.show()
print("Saved baseline_comparison.pdf/png")

Saved baseline_comparison.pdf/png


/var/folders/fx/gjhbmrbj5jn295_9wrqpbsv80000gn/T/ipykernel_47066/1929928194.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
